# 🔗 RAG 및 계획 하네스 구축

이 노트북은 τ-Knowledge banking_knowledge를 위한 RAG(검색 증강 생성) 시스템과
LangGraph 기반 계획 하네스를 구축하고 테스트합니다.

## 아키텍처

```
사용자 요청 → 계획 수립 → KB 검색/도구 발견 → 정책 적용 →
도구 호출 → 결과 검증 → 응답/후속 질문
```

## 구성 요소

| 구성 요소 | 역할 |
|----------|------|
| **KB 인덱스** | 정책·제품·절차 문서 벡터 검색 |
| **예제 인덱스** | 학습 데이터의 추론 패턴 참고 (선택) |
| **BM25** | 키워드 기반 검색 (진단 기준선) |
| **LangGraph** | 상태 기반 계획 및 실행 그래프 |
| **FastAPI** | `/v1/agent/step`, `/v1/qa`, `/healthz` 엔드포인트 |
| **τ 어댑터** | 공식 시뮬레이션 도구 연결 |

### 두 가지 모드
- **`simple_rag`**: 턴당 고정 검색 → 응답 (추가 계획/수리 없음)
- **`agent_rag`**: 명시적 계획 + 추가 검색 + 검증 루프

In [ ]:
"""환경 확인 — 00_preflight.ipynb 에서 이미 설치 완료."""

import os
from pathlib import Path

_nb_dir = Path.cwd()
_project_root = _nb_dir
for _p in [_nb_dir] + list(_nb_dir.parents):
    if (_p / "pyproject.toml").exists():
        _project_root = _p
        break

os.environ["RHOAI_PROJECT_ROOT"] = str(_project_root)

try:
    import rhoai_model_training_lab.config as _cfg
    _cfg.PROJECT_ROOT = _project_root
    print(f"✅ 프로젝트 루트: {_project_root}")
except ImportError:
    raise ImportError(
        "❌ 패키지 미설치 — 먼저 00_preflight.ipynb 를 실행하세요."
    )


In [ ]:
"""Build KB index from bundle."""

import os
from pathlib import Path

from rhoai_model_training_lab.config import (
    load_env, load_rag_config, load_bundle_config, PROJECT_ROOT,
)
from rhoai_model_training_lab.data import BundleManager

load_env()

rag_config = load_rag_config()
release_config = load_bundle_config()
bundle_base = release_config.get("bundle", {}).get("base_path", "data/prepared/tau-knowledge-v1")
bundle_path = PROJECT_ROOT / bundle_base

mgr = BundleManager.load_bundle(bundle_path)
kb_docs = mgr.get_kb_documents()
print(f"KB 문서 수: {len(kb_docs)}")

# Build vector index
embedding_model_id = rag_config["retrieval"]["embedding"]["model_id"]
persist_dir = PROJECT_ROOT / rag_config["retrieval"]["vector_store"]["persist_directory"]
chunk_size = rag_config["retrieval"]["kb_index"]["chunk_size"]
chunk_overlap = rag_config["retrieval"]["kb_index"]["chunk_overlap"]

print(f"\n임베딩 모델: {embedding_model_id}")
print(f"벡터 저장소: {persist_dir}")
print(f"청크 크기: {chunk_size}, 오버랩: {chunk_overlap}")

# Initialize embedding model
from sentence_transformers import SentenceTransformer

print("\n임베딩 모델 로딩 중...")
embed_model = SentenceTransformer(embedding_model_id)
embedding_dim = embed_model.get_sentence_embedding_dimension()
print(f"  임베딩 차원: {embedding_dim}")

# Initialize ChromaDB
import chromadb

persist_dir.mkdir(parents=True, exist_ok=True)
chroma_client = chromadb.PersistentClient(path=str(persist_dir))

# Create/reset KB collection
kb_collection_name = rag_config["retrieval"]["kb_index"]["collection_name"]
try:
    chroma_client.delete_collection(kb_collection_name)
except Exception:
    pass

kb_collection = chroma_client.create_collection(
    name=kb_collection_name,
    metadata={"hnsw:space": "cosine"},
)

# Chunk and index documents
print("\n문서 청킹 및 인덱싱 중...")
total_chunks = 0
for doc in kb_docs:
    text = doc.get("text", "")
    doc_id = doc.get("document_id", doc.get("id", ""))

    # Simple chunking with overlap
    chunks = []
    for start in range(0, len(text), chunk_size - chunk_overlap):
        chunk_text = text[start:start + chunk_size]
        if chunk_text.strip():
            chunks.append(chunk_text)

    if not chunks:
        continue

    embeddings = embed_model.encode(chunks, show_progress_bar=False).tolist()

    ids = [f"{doc_id}_chunk_{i}" for i in range(len(chunks))]
    metadatas = [{"document_id": doc_id, "chunk_index": i} for i in range(len(chunks))]

    kb_collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=chunks,
        metadatas=metadatas,
    )
    total_chunks += len(chunks)

print(f"\n✅ KB 인덱스 구축 완료")
print(f"   총 문서: {len(kb_docs)}")
print(f"   총 청크: {total_chunks}")
print(f"   저장 위치: {persist_dir}")

In [ ]:
"""Test retrieval — dense search and BM25 comparison."""

from rank_bm25 import BM25Okapi

# Test queries
test_queries = [
    "What is the maximum daily transfer limit?",
    "How do I dispute a transaction?",
    "What are the requirements for opening a premium account?",
]

print("=" * 70)
print("🔍 검색 테스트: Dense (벡터) vs BM25 (키워드) 비교")
print("=" * 70)

# Build BM25 index for comparison
all_chunks = kb_collection.get(include=["documents"])
chunk_texts = all_chunks["documents"]
chunk_ids = all_chunks["ids"]

tokenized_corpus = [doc.lower().split() for doc in chunk_texts]
bm25 = BM25Okapi(tokenized_corpus)

for query in test_queries:
    print(f"\n📝 질의: \"{query}\"")

    # Dense retrieval
    query_embedding = embed_model.encode([query]).tolist()
    dense_results = kb_collection.query(
        query_embeddings=query_embedding,
        n_results=3,
    )

    print("\n  🔷 Dense 검색 결과 (상위 3):")
    for i, (doc, dist) in enumerate(
        zip(dense_results["documents"][0], dense_results["distances"][0])
    ):
        score = 1 - dist  # cosine similarity
        print(f"    {i+1}. [유사도: {score:.3f}] {doc[:120]}...")

    # BM25 retrieval
    bm25_scores = bm25.get_scores(query.lower().split())
    top_indices = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:3]

    print("\n  🔶 BM25 검색 결과 (상위 3):")
    for rank, idx in enumerate(top_indices):
        print(f"    {rank+1}. [점수: {bm25_scores[idx]:.3f}] {chunk_texts[idx][:120]}...")

    print(f"  {'─' * 60}")

In [ ]:
"""Start backend and test /healthz."""

import subprocess
import sys
import time
import httpx

backend_host = os.environ.get("BACKEND_HOST", "127.0.0.1")
backend_port = int(os.environ.get("BACKEND_PORT", "8000"))
backend_url = f"http://{backend_host}:{backend_port}"

print("=" * 70)
print("🚀 백엔드 시작 및 헬스 체크")
print("=" * 70)

# Check if already running
try:
    resp = httpx.get(f"{backend_url}/healthz", timeout=5)
    if resp.status_code == 200:
        print(f"✅ 백엔드가 이미 실행 중입니다: {backend_url}")
        health = resp.json()
        print(f"   상태: {health}")
        backend_running = True
    else:
        backend_running = False
except Exception:
    backend_running = False

if not backend_running:
    # Try starting via script
    start_script = PROJECT_ROOT / "scripts" / "start_backend.sh"
    if start_script.exists():
        print(f"\n백엔드 시작 중... (포트: {backend_port})")
        proc = subprocess.Popen(
            ["bash", str(start_script), "--host", backend_host, "--port", str(backend_port)],
            cwd=str(PROJECT_ROOT),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
        )

        # Wait for startup
        for i in range(30):
            time.sleep(1)
            try:
                resp = httpx.get(f"{backend_url}/healthz", timeout=3)
                if resp.status_code == 200:
                    print(f"\n✅ 백엔드 시작 완료: {backend_url}")
                    backend_running = True
                    break
            except Exception:
                print(".", end="", flush=True)
    else:
        print("\n⚠️  시작 스크립트를 찾을 수 없습니다.")
        print("   수동 시작: bash scripts/start_backend.sh")
        print("   또는 uvicorn으로 직접 시작:")
        print(f"   uvicorn rhoai_model_training_lab.api:app --host {backend_host} --port {backend_port}")

if not backend_running:
    print("\n⚠️  백엔드가 시작되지 않았습니다.")
    print("   이후 셀은 백엔드 실행 없이 건너뛸 수 있습니다.")

In [ ]:
"""Test /v1/qa endpoint."""

print("=" * 70)
print("🧪 /v1/qa 엔드포인트 테스트")
print("=" * 70)

qa_tests = [
    {
        "question": "What is the daily transfer limit for standard accounts?",
        "mode": "rag",
    },
    {
        "question": "How do I report a stolen card?",
        "mode": "rag",
    },
]

if backend_running:
    for i, test in enumerate(qa_tests, 1):
        print(f"\n--- 테스트 {i}: {test['question'][:50]}... ---")
        try:
            resp = httpx.post(
                f"{backend_url}/v1/qa",
                json=test,
                timeout=30,
            )
            resp.raise_for_status()
            result = resp.json()

            print(f"  상태: {result.get('status', 'N/A')}")
            print(f"  응답: {result.get('answer', 'N/A')[:200]}")
            if result.get("citations"):
                print(f"  인용: {len(result['citations'])}건")
                for cit in result["citations"][:2]:
                    print(f"    - {cit.get('document_id', 'N/A')}: {cit.get('text_excerpt', '')[:80]}")
            if result.get("usage"):
                usage = result["usage"]
                print(f"  사용량: {usage.get('total_tokens', 0)} 토큰")
                print(f"  검색 지연: {usage.get('retrieval_latency_ms', 0):.0f}ms")
        except Exception as exc:
            print(f"  ❌ 실패: {exc}")
else:
    print("⏭️ 백엔드가 실행되지 않아 건너뜁니다.")

In [ ]:
"""Test /v1/agent/step with a sample scenario."""

print("=" * 70)
print("🧪 /v1/agent/step 에이전트 스텝 테스트")
print("=" * 70)

if backend_running:
    # Sample banking scenario
    agent_request = {
        "session_id": "test-session-001",
        "request_id": "step-001",
        "messages": [
            {"role": "user", "content": "I want to transfer $5000 to account 987654. Is that within my daily limit?"},
        ],
        "available_tools": [
            {
                "type": "function",
                "function": {
                    "name": "get_account_info",
                    "description": "Get account information including balance and limits",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "account_id": {"type": "string"}
                        },
                        "required": ["account_id"],
                    },
                },
            },
            {
                "type": "function",
                "function": {
                    "name": "transfer_funds",
                    "description": "Transfer funds between accounts",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "from_account": {"type": "string"},
                            "to_account": {"type": "string"},
                            "amount": {"type": "number"},
                        },
                        "required": ["from_account", "to_account", "amount"],
                    },
                },
            },
        ],
        "mode": "agent_rag",
        "knowledge_access": "rag",
    }

    try:
        resp = httpx.post(
            f"{backend_url}/v1/agent/step",
            json=agent_request,
            timeout=60,
        )
        resp.raise_for_status()
        result = resp.json()

        print(f"\n  세션 ID: {result.get('session_id')}")
        print(f"  상태: {result.get('status')}")

        if result.get("plan"):
            print(f"\n  📋 계획: {result['plan'][:300]}")

        if result.get("message"):
            msg = result["message"]
            print(f"\n  🤖 응답 [{msg.get('role', '?')}]: {msg.get('content', '')[:300]}")

        if result.get("tool_calls"):
            print(f"\n  🔧 도구 호출:")
            for tc in result["tool_calls"]:
                fn = tc.get("function", tc)
                print(f"    → {fn.get('name', '?')}({json.dumps(fn.get('arguments', {}), ensure_ascii=False)[:100]})")

        if result.get("citations"):
            print(f"\n  📎 인용: {len(result['citations'])}건")

        print(f"\n  추적 ID: {result.get('trace_id', 'N/A')}")

    except Exception as exc:
        print(f"\n  ❌ 실패: {exc}")
else:
    print("⏭️ 백엔드가 실행되지 않아 건너뜁니다.")

In [ ]:
"""Compare simple_rag vs agent_rag modes."""

import json

print("=" * 70)
print("⚖️ simple_rag vs agent_rag 모드 비교")
print("=" * 70)

comparison_query = {
    "session_id": "compare-test",
    "messages": [
        {"role": "user", "content": "I need to check if I can get a premium credit card upgrade and what the requirements are."},
    ],
    "knowledge_access": "rag",
}

if backend_running:
    for mode in ["simple_rag", "agent_rag"]:
        print(f"\n{'─' * 60}")
        print(f"📋 모드: {mode}")
        print(f"{'─' * 60}")

        request = {**comparison_query, "mode": mode, "request_id": f"cmp-{mode}"}

        try:
            resp = httpx.post(
                f"{backend_url}/v1/agent/step",
                json=request,
                timeout=60,
            )
            resp.raise_for_status()
            result = resp.json()

            if result.get("plan"):
                print(f"  계획: {result['plan'][:200]}")
            else:
                print(f"  계획: (없음 — simple_rag에서는 정상)")

            if result.get("message"):
                content = result["message"].get("content", "")
                print(f"  응답: {content[:300]}")

            if result.get("tool_calls"):
                print(f"  도구 호출: {len(result['tool_calls'])}건")

            if result.get("usage"):
                usage = result["usage"]
                print(f"  토큰: {usage.get('total_tokens', 0)}")
                print(f"  총 지연: {usage.get('total_latency_ms', 0):.0f}ms")

        except Exception as exc:
            print(f"  ❌ {exc}")

    print(f"\n{'=' * 70}")
    print("비교 요약:")
    print("  • simple_rag: 턴당 고정 검색, 추가 계획 없음 → 빠르지만 단순")
    print("  • agent_rag: 명시적 계획, 추가 검색, 검증 루프 → 느리지만 정교")
    print("  • 공식 평가에서 두 모드의 성능을 비교합니다.")
else:
    print("⏭️ 백엔드가 실행되지 않아 건너뜁니다.")
    print("\n📝 모드 비교 설명:")
    print("  • simple_rag: 턴당 고정 검색 → 응답/도구 호출 (계획 없음)")
    print("  • agent_rag: 계획 수립 → 검색 → 정책 적용 → 도구 호출 → 검증")
    print("  • 동일 모델, 동일 리트리버, 동일 비즈니스 도구 사용")
    print("  • agent_rag만 명시적 계획과 추가 검색을 허용")

print("\n다음 단계:")
print("  📓 07_evaluate.ipynb — 평가 실행")